# إعادة اختبارات NASA C-MAPSS — المرحلة الأولى
## فحص بيانات FD001 وFD004 قبل تدريب النماذج

**طريقة الاستخدام:** افتح هذا الملف في Google Colab ثم اختر `Runtime` → `Run all` (أو شغّل الخلايا بالترتيب).

هذه المرحلة لا تعيد إنتاج نتائج البحث بعد؛ إنها تنزّل بيانات NASA وتتحقق من سلامتها وأعداد المحركات والدورات. لا ترفع أي بيانات شخصية هنا.

المصدر: NASA Prognostics Center of Excellence — Turbofan Engine Degradation Simulation Data Set.


In [1]:
# 1) تنزيل مجموعة البيانات الأصلية من مستودع NASA
import os, zipfile, urllib.request, pathlib

DATA_DIR = "/content/cmapss_data"
ZIP_PATH = "/content/cmapss.zip"
os.makedirs(DATA_DIR, exist_ok=True)

url = "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip"

if not os.path.exists(ZIP_PATH):
    print("جاري تنزيل ملف NASA... قد يستغرق ذلك بضع دقائق.")
    urllib.request.urlretrieve(url, ZIP_PATH)
    print("اكتمل التنزيل.")
else:
    print("ملف ZIP موجود مسبقًا.")

print("حجم الملف بالميغابايت:", round(os.path.getsize(ZIP_PATH)/1024**2, 2))


جاري تنزيل ملف NASA... قد يستغرق ذلك بضع دقائق.
اكتمل التنزيل.
حجم الملف بالميغابايت: 11.85


In [2]:
# 2) فك الضغط وعرض الملفات
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(DATA_DIR)

all_txt = []
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith(".txt"):
            all_txt.append(os.path.join(root, f))

print("عدد ملفات TXT:", len(all_txt))
for f in sorted(all_txt):
    print(os.path.relpath(f, DATA_DIR))


عدد ملفات TXT: 0


In [3]:
# 3) فك ضغط الملفات المتداخلة (إن وجدت) والعثور على ملفات FD001 وFD004 وقراءتها
import pandas as pd
import zipfile
import os
from pathlib import Path

# فحص إذا كان هناك ملف مضغوط داخلي وفك ضغطه
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if f.endswith(".zip"):
            zip_inner_path = os.path.join(root, f)
            print(f"جاري فك ضغط الملف الداخلي: {f}")
            with zipfile.ZipFile(zip_inner_path, "r") as z:
                z.extractall(DATA_DIR)

def locate_file(filename):
    matches = list(Path(DATA_DIR).rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"لم أجد {filename}. راجع قائمة الملفات في الخلية السابقة."
        )
    return str(matches[0])

COLS = ["unit", "cycle"] + [f"setting_{i}" for i in range(1,4)] + [f"sensor_{i}" for i in range(1,22)]

data = {}
for subset in ["FD001", "FD004"]:
    for split in ["train", "test"]:
        fname = f"{split}_{subset}.txt"
        path = locate_file(fname)
        df = pd.read_csv(path, sep=r"\s+", header=None, names=COLS)
        data[(subset, split)] = df
        print(fname, "shape =", df.shape, "| engines =", df.unit.nunique())

    rul_path = locate_file(f"RUL_{subset}.txt")
    rul = pd.read_csv(rul_path, sep=r"\s+", header=None, names=["RUL"])
    data[(subset, "RUL")] = rul
    print(f"RUL_{subset}.txt shape =", rul.shape)
    print("-" * 55)

جاري فك ضغط الملف الداخلي: CMAPSSData.zip
train_FD001.txt shape = (20631, 26) | engines = 100
test_FD001.txt shape = (13096, 26) | engines = 100
RUL_FD001.txt shape = (100, 1)
-------------------------------------------------------
train_FD004.txt shape = (61249, 26) | engines = 249
test_FD004.txt shape = (41214, 26) | engines = 248
RUL_FD004.txt shape = (248, 1)
-------------------------------------------------------


In [4]:
# 4) فحوصات أساسية: القيم المفقودة، القيم غير المنتهية، وأطوال المسارات
import numpy as np

for subset in ["FD001", "FD004"]:
    print(f"\n===== {subset} =====")
    for split in ["train", "test"]:
        df = data[(subset, split)]
        lengths = df.groupby("unit")["cycle"].agg(["min", "max", "count"])
        print(
            split,
            "| missing =", int(df.isna().sum().sum()),
            "| non-finite =", int((~np.isfinite(df.select_dtypes(include=[np.number])).to_numpy()).sum()),
            "| engines =", df.unit.nunique(),
            "| rows =", len(df),
            "| cycle min/max =", int(df.cycle.min()), int(df.cycle.max()),
            "| median cycles/engine =", float(lengths["count"].median())
        )
    print("عدد قيم RUL الرسمية =", len(data[(subset, "RUL")]))
    print("عدد محركات الاختبار =", data[(subset, "test")].unit.nunique())



===== FD001 =====
train | missing = 0 | non-finite = 0 | engines = 100 | rows = 20631 | cycle min/max = 1 362 | median cycles/engine = 199.0
test | missing = 0 | non-finite = 0 | engines = 100 | rows = 13096 | cycle min/max = 1 303 | median cycles/engine = 133.5
عدد قيم RUL الرسمية = 100
عدد محركات الاختبار = 100

===== FD004 =====
train | missing = 0 | non-finite = 0 | engines = 249 | rows = 61249 | cycle min/max = 1 543 | median cycles/engine = 234.0
test | missing = 0 | non-finite = 0 | engines = 248 | rows = 41214 | cycle min/max = 1 486 | median cycles/engine = 153.5
عدد قيم RUL الرسمية = 248
عدد محركات الاختبار = 248


## المرحلة الثانية: تجهيز وحساب قيم RUL المستهدفة (Target RUL)

في بيانات التدريب، لا يتم توفير قيم RUL جاهزة لكل دورة. نقوم بحسابها كالتالي:
$$\text{RUL}_{t} = \text{Max Cycle} - t$$
حيث $t$ هو رقم الدورة الحالية لملف التدريب للمحرك المعني.

سنقوم أيضاً ببرمجة خيار **Piecewise RUL** (تحديد الحد الأقصى للـ RUL عند قيمة ثابتة مثل 125 دورة) وهو الأسلوب القياسي المعتمد في الأبحاث لمنع النموذج من محاولة التنبؤ بقيم RUL مرتفعة جداً وغير واقعية في البداية عندما يكون المحرك سليماً تماماً.

In [5]:
# 5) دالة لحساب RUL لبيانات التدريب (مع خيار Piecewise RUL)
def add_remaining_useful_life(df, max_rul=125):
    # حساب أقصى دورة لكل محرك
    max_cycle = df.groupby("unit")["cycle"].transform("max")
    # الحساب الخطي التقليدي
    df["RUL_linear"] = max_cycle - df["cycle"]

    # تطبيق Piecewise RUL (الحد الأقصى)
    if max_rul is not None:
        df["RUL"] = df["RUL_linear"].clip(upper=max_rul)
    else:
        df["RUL"] = df["RUL_linear"]
    return df

# تطبيق الدالة على مجموعات التدريب
for subset in ["FD001", "FD004"]:
    train_df = data[(subset, "train")]
    data[(subset, "train")] = add_remaining_useful_life(train_df, max_rul=125)
    print(f"تمت إضافة عمود RUL لـ {subset}_train. شكل البيانات الحالي:", data[(subset, "train")].shape)

# عرض عينة للمحرك الأول في FD001 للتأكد من صحة الحسابات
display(data[("FD001", "train")][data[("FD001", "train")].unit == 1][["unit", "cycle", "RUL_linear", "RUL"]].head(15))

تمت إضافة عمود RUL لـ FD001_train. شكل البيانات الحالي: (20631, 28)
تمت إضافة عمود RUL لـ FD004_train. شكل البيانات الحالي: (61249, 28)


,unit,cycle,RUL_linear,RUL
0,1,1,191,125
1,1,2,190,125
2,1,3,189,125
3,1,4,188,125
4,1,5,187,125
5,1,6,186,125
6,1,7,185,125
7,1,8,184,125
8,1,9,183,125
9,1,10,182,125


In [6]:
for name, df in {
    "FD001_train": data[("FD001", "train")],
    "FD004_train": data[("FD004", "train")]
}.items():

    print("\n", name)
    print("RUL min:", df["RUL"].min())
    print("RUL max:", df["RUL"].max())
    print("RUL_linear max:", df["RUL_linear"].max())
    print("Number of RUL values > 125:",
          (df["RUL_linear"] > 125).sum())
    print("Missing values:", df.isnull().sum().sum())


 FD001_train
RUL min: 0
RUL max: 125
RUL_linear max: 361
Number of RUL values > 125: 8031
Missing values: 0

 FD004_train
RUL min: 0
RUL max: 125
RUL_linear max: 542
Number of RUL values > 125: 29875
Missing values: 0


In [7]:
# فحص ملفات الاختبار وملفات RUL الرسمية

for subset in ["FD001", "FD004"]:

    test_df = data[(subset, "test")]
    rul_df = data[(subset, "RUL")]

    print("\n======", subset, "======")
    print("Test shape:", test_df.shape)
    print("Number of test engines:", test_df["unit"].nunique())
    print("RUL file shape:", rul_df.shape)
    print("RUL missing values:", rul_df.isnull().sum().sum())

    print("First 10 official RUL values:")
    print(rul_df.head(10).T)


====== FD001 ======
Test shape: (13096, 26)
Number of test engines: 100
RUL file shape: (100, 1)
RUL missing values: 0
First 10 official RUL values:
       0   1   2   3   4   5   6   7    8   9
RUL  112  98  69  82  91  93  91  95  111  96

====== FD004 ======
Test shape: (41214, 26)
Number of test engines: 248
RUL file shape: (248, 1)
RUL missing values: 0
First 10 official RUL values:
      0   1    2   3    4   5   6   7   8    9
RUL  22  39  107  75  149  78  94  14  99  162


In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# إعدادات التجربة الأولية
SEQ_LEN = 30
FEATURE_COLS = (
    [f"setting_{i}" for i in range(1, 4)] +
    [f"sensor_{i}" for i in range(1, 22)]
)

def prepare_lstm_data(subset):

    train_df = data[(subset, "train")].copy()
    test_df = data[(subset, "test")].copy()
    test_rul = data[(subset, "RUL")]["RUL"].to_numpy()

    # فصل المحركات، وليس الصفوف، لمنع تسرب البيانات
    units = train_df["unit"].unique()
    train_units, val_units = train_test_split(
        units, test_size=0.20, random_state=42
    )

    # تطبيع الخصائص باستخدام بيانات محركات التدريب فقط
    scaler = StandardScaler()
    scaler.fit(
        train_df[train_df["unit"].isin(train_units)][FEATURE_COLS]
    )

    train_df[FEATURE_COLS] = scaler.transform(train_df[FEATURE_COLS])
    test_df[FEATURE_COLS] = scaler.transform(test_df[FEATURE_COLS])

    def make_train_windows(df, selected_units):
        X, y = [], []

        for unit in selected_units:
            engine = df[df["unit"] == unit].sort_values("cycle")
            values = engine[FEATURE_COLS].to_numpy()
            labels = engine["RUL"].to_numpy()

            for end in range(SEQ_LEN - 1, len(engine)):
                X.append(values[end - SEQ_LEN + 1:end + 1])
                y.append(labels[end])

        return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)

    def make_test_windows(df):
        X = []

        for unit in sorted(df["unit"].unique()):
            engine = df[df["unit"] == unit].sort_values("cycle")
            values = engine[FEATURE_COLS].to_numpy()

            # إذا كان عدد الدورات أقل من SEQ_LEN، نقوم بحشوها بالأصفار (Zero-padding) من الأعلى
            if len(values) < SEQ_LEN:
                padding_size = SEQ_LEN - len(values)
                padding = np.zeros((padding_size, values.shape[1]), dtype=values.dtype)
                values = np.vstack((padding, values))
                print(f"تنبيه: تم عمل حشوة (Padding) للمحرك رقم {unit} بمقدار {padding_size} أسطر صفريّة.")

            X.append(values[-SEQ_LEN:])

        return np.asarray(X, dtype=np.float32)

    X_train, y_train = make_train_windows(train_df, train_units)
    X_val, y_val = make_train_windows(train_df, val_units)
    X_test = make_test_windows(test_df)

    assert len(X_test) == len(test_rul)

    print(f"\n{subset}")
    print("X_train:", X_train.shape, "y_train:", y_train.shape)
    print("X_val:", X_val.shape, "y_val:", y_val.shape)
    print("X_test:", X_test.shape)
    print("Official test RUL:", test_rul.shape)

    return X_train, y_train, X_val, y_val, X_test, test_rul, scaler


fd001_data = prepare_lstm_data("FD001")
fd004_data = prepare_lstm_data("FD004")


FD001
X_train: (14241, 30, 24) y_train: (14241,)
X_val: (3490, 30, 24) y_val: (3490,)
X_test: (100, 30, 24)
Official test RUL: (100,)
تنبيه: تم عمل حشوة (Padding) للمحرك رقم 10 بمقدار 7 أسطر صفريّة.
تنبيه: تم عمل حشوة (Padding) للمحرك رقم 19 بمقدار 6 أسطر صفريّة.
تنبيه: تم عمل حشوة (Padding) للمحرك رقم 28 بمقدار 10 أسطر صفريّة.
تنبيه: تم عمل حشوة (Padding) للمحرك رقم 125 بمقدار 9 أسطر صفريّة.
تنبيه: تم عمل حشوة (Padding) للمحرك رقم 141 بمقدار 11 أسطر صفريّة.
تنبيه: تم عمل حشوة (Padding) للمحرك رقم 156 بمقدار 2 أسطر صفريّة.
تنبيه: تم عمل حشوة (Padding) للمحرك رقم 164 بمقدار 1 أسطر صفريّة.
تنبيه: تم عمل حشوة (Padding) للمحرك رقم 204 بمقدار 11 أسطر صفريّة.
تنبيه: تم عمل حشوة (Padding) للمحرك رقم 229 بمقدار 10 أسطر صفريّة.
تنبيه: تم عمل حشوة (Padding) للمحرك رقم 239 بمقدار 2 أسطر صفريّة.
تنبيه: تم عمل حشوة (Padding) للمحرك رقم 246 بمقدار 1 أسطر صفريّة.

FD004
X_train: (43523, 30, 24) y_train: (43523,)
X_val: (10505, 30, 24) y_val: (10505,)
X_test: (248, 30, 24)
Official test RUL: (248,)


In [16]:
X_train, y_train, X_val, y_val, X_test, y_test, scaler = fd004_data

print("FD004 data restored successfully")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

FD004 data restored successfully
X_train: (43523, 30, 24)
X_val: (10505, 30, 24)
X_test: (248, 30, 24)
y_test: (248,)


In [17]:
print("build_lstm exists:", "build_lstm" in globals())
print("EarlyStopping exists:", "EarlyStopping" in globals())
print("X_train shape:", X_train.shape)

build_lstm exists: False
EarlyStopping exists: False
X_train shape: (43523, 30, 24)


In [18]:
import tensorflow as tf
import numpy as np
import random
import os

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def build_lstm():
    model = Sequential([
        Input(shape=(30, 24)),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"]
    )

    return model

print("LSTM model functions are ready")

LSTM model functions are ready


In [19]:
set_seed(42)

model = build_lstm()

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        22,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,745 (139.63 KB)

 Trainable params: 35,745 (139.63 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
set_seed(42)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

print("Training completed")
print("Epochs:", len(history.history["loss"]))
print("Best validation MAE:",
      min(history.history["val_mae"]))
print("Best validation loss:",
      min(history.history["val_loss"]))

Epoch 1/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 28s 68ms/step - loss: 5213.7739 - mae: 62.4357 - val_loss: 1909.0212 - val_mae: 39.8855
Epoch 2/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 21s 62ms/step - loss: 1793.6285 - mae: 37.6244 - val_loss: 1650.6614 - val_mae: 36.4725
Epoch 3/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 43s 67ms/step - loss: 723.9831 - mae: 21.6048 - val_loss: 576.6559 - val_mae: 19.0569
Epoch 4/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 21s 62ms/step - loss: 463.2684 - mae: 16.1705 - val_loss: 521.0187 - val_mae: 17.3932
Epoch 5/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 23s 68ms/step - loss: 412.4548 - mae: 15.1865 - val_loss: 471.8784 - val_mae: 16.0196
Epoch 6/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 22s 64ms/step - loss: 385.7344 - mae: 14.6286 - val_loss: 472.0888 - val_mae: 15.6709
Epoch 7/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 42s 67ms/step - loss: 355.3798 - mae: 14.0420 - val_loss: 458.3313 - val_mae: 15.1877
Epoch 8/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 21s 63ms/step - loss: 332.7932 - mae: 13.5686 - val_loss: 500.8596 - val_m

In [21]:
import pandas as pd

# حفظ أفضل نموذج بعد Early Stopping
model.save("FD004_LSTM_RUL125.keras")

# حفظ سجل التدريب
history_df = pd.DataFrame(history.history)
history_df.to_csv("FD004_LSTM_Training_History.csv", index=False)

print("Model and training history saved successfully")

Model and training history saved successfully


In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# التنبؤ باستخدام أفضل أوزان النموذج المحفوظة
y_pred = model.predict(X_test, verbose=0).flatten()

# القيم الحقيقية والمتوقعة
y_true = np.asarray(y_test).flatten()

# حساب المقاييس
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# NASA Score
errors = y_pred - y_true

scores = np.where(
    errors < 0,
    np.exp(-errors / 13) - 1,
    np.exp(errors / 10) - 1
)

nasa_score = np.sum(scores)

print("FD004 Final Test Results")
print("------------------------")
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("NASA Score:", round(nasa_score, 4))
print("Number of test engines:", len(y_true))

FD004 Final Test Results
------------------------
MAE: 23.6127
RMSE: 31.1272
NASA Score: 13500.1266
Number of test engines: 248


In [23]:
import pandas as pd

fd004_results = pd.DataFrame([{
    "Dataset": "FD004",
    "Model": "LSTM",
    "RUL_Cap": 125,
    "Seed": 42,
    "MAE": mae,
    "RMSE": rmse,
    "NASA_Score": nasa_score,
    "Test_Engines": len(y_true)
}])

fd004_results.to_csv(
    "FD004_Final_Test_Results.csv",
    index=False
)

print(fd004_results)
print("Results saved successfully")

  Dataset Model  RUL_Cap  Seed        MAE       RMSE    NASA_Score  \
0   FD004  LSTM      125    42  23.612747  31.127177  13500.126573   

   Test_Engines  
0           248  
Results saved successfully


In [24]:
# تجهيز بيانات FD001
X_train_1, y_train_1, X_val_1, y_val_1, X_test_1, y_test_1, scaler_1 = fd001_data

print("FD001 data ready")
print("X_train:", X_train_1.shape)
print("X_val:", X_val_1.shape)
print("X_test:", X_test_1.shape)
print("y_test:", y_test_1.shape)

FD001 data ready
X_train: (14241, 30, 24)
X_val: (3490, 30, 24)
X_test: (100, 30, 24)
y_test: (100,)


In [26]:
set_seed(42)

model_fd001 = build_lstm()

early_stop_fd001 = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history_fd001 = model_fd001.fit(
    X_train_1,
    y_train_1,
    validation_data=(X_val_1, y_val_1),
    epochs=50,
    batch_size=128,
    callbacks=[early_stop_fd001],
    verbose=1
)

print("FD001 Training completed")
print("Epochs:", len(history_fd001.history["loss"]))
print("Best validation MAE:",
      min(history_fd001.history["val_mae"]))
print("Best validation loss:",
      min(history_fd001.history["val_loss"]))

Epoch 1/50



KeyboardInterrupt



In [27]:
print("FD001 data:", X_train_1.shape)
print("Model exists:", "model_fd001" in globals())
print("Runtime ready")

FD001 data: (14241, 30, 24)
Model exists: True
Runtime ready


In [28]:
set_seed(42)

model_fd001 = build_lstm()

early_stop_fd001 = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history_fd001 = model_fd001.fit(
    X_train_1,
    y_train_1,
    validation_data=(X_val_1, y_val_1),
    epochs=50,
    batch_size=128,
    callbacks=[early_stop_fd001],
    verbose=1
)

print("FD001 Training completed")
print("Epochs:", len(history_fd001.history["loss"]))
print("Best validation MAE:",
      min(history_fd001.history["val_mae"]))
print("Best validation loss:",
      min(history_fd001.history["val_loss"]))

Epoch 1/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 13s 68ms/step - loss: 7369.6089 - mae: 75.4672 - val_loss: 6205.7676 - val_mae: 68.3541
Epoch 2/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 8s 74ms/step - loss: 5152.0229 - mae: 61.4128 - val_loss: 3943.4839 - val_mae: 52.9970
Epoch 3/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 9s 76ms/step - loss: 3054.9253 - mae: 46.6648 - val_loss: 2215.5396 - val_mae: 40.4337
Epoch 4/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 10s 74ms/step - loss: 1590.6450 - mae: 33.5411 - val_loss: 904.3607 - val_mae: 24.5929
Epoch 5/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 8s 74ms/step - loss: 725.6998 - mae: 22.3371 - val_loss: 463.6916 - val_mae: 18.0222
Epoch 6/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 7s 58ms/step - loss: 475.3892 - mae: 17.8233 - val_loss: 303.2250 - val_mae: 14.2098
Epoch 7/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 8s 75ms/step - loss: 378.8379 - mae: 15.5309 - val_loss: 265.9113 - val_mae: 12.9232
Epoch 8/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 7s 58ms/step - loss: 327.2607 - mae: 14.0627 - val_loss: 189.3468 - val_mae:

In [29]:
model_fd001.save("FD001_LSTM_RUL125.keras")

import pandas as pd

history_fd001_df = pd.DataFrame(history_fd001.history)
history_fd001_df.to_csv(
    "FD001_LSTM_Training_History.csv",
    index=False
)

print("FD001 model and history saved successfully")

FD001 model and history saved successfully


In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# التنبؤ باستخدام أفضل أوزان النموذج
y_pred_1 = model_fd001.predict(X_test_1, verbose=0).flatten()

y_true_1 = np.asarray(y_test_1).flatten()

# حساب المقاييس
mae_1 = mean_absolute_error(y_true_1, y_pred_1)
rmse_1 = np.sqrt(mean_squared_error(y_true_1, y_pred_1))

# NASA Score
errors_1 = y_pred_1 - y_true_1

scores_1 = np.where(
    errors_1 < 0,
    np.exp(-errors_1 / 13) - 1,
    np.exp(errors_1 / 10) - 1
)

nasa_score_1 = np.sum(scores_1)

print("FD001 Final Test Results")
print("------------------------")
print("MAE:", round(mae_1, 4))
print("RMSE:", round(rmse_1, 4))
print("NASA Score:", round(nasa_score_1, 4))
print("Number of test engines:", len(y_true_1))

FD001 Final Test Results
------------------------
MAE: 12.1151
RMSE: 16.0104
NASA Score: 418.2675
Number of test engines: 100


In [31]:
import pandas as pd

fd001_results = pd.DataFrame([{
    "Dataset": "FD001",
    "Model": "LSTM",
    "RUL_Cap": 125,
    "Seed": 42,
    "MAE": mae_1,
    "RMSE": rmse_1,
    "NASA_Score": nasa_score_1,
    "Test_Engines": len(y_true_1)
}])

fd001_results.to_csv(
    "FD001_Final_Test_Results.csv",
    index=False
)

print(fd001_results)
print("FD001 results saved successfully")

  Dataset Model  RUL_Cap  Seed        MAE       RMSE  NASA_Score  Test_Engines
0   FD001  LSTM      125    42  12.115134  16.010444  418.267476           100
FD001 results saved successfully


In [32]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, RepeatVector,
    TimeDistributed, Dense, Dropout
)

def build_lstm_autoencoder():
    inputs = Input(shape=(30, 24))

    # Encoder
    encoded = LSTM(64, return_sequences=False)(inputs)
    encoded = Dropout(0.2)(encoded)

    # Bottleneck
    latent = Dense(32, activation="relu")(encoded)

    # Regression head for RUL prediction
    outputs = Dense(1, name="rul_prediction")(latent)

    model = Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"]
    )

    return model

autoencoder_model = build_lstm_autoencoder()

autoencoder_model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 30, 24)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ (None, 64)             │        22,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rul_prediction (Dense)          │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,897 (97.25 KB)

 Trainable params: 24,897 (97.25 KB)

 Non-trainable params: 0 (0.00 B)

In [33]:
set_seed(42)

early_stop_ae = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history_ae_fd004 = autoencoder_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=128,
    callbacks=[early_stop_ae],
    verbose=1
)

print("FD004 Autoencoder training completed")
print("Epochs:", len(history_ae_fd004.history["loss"]))
print("Best validation MAE:",
      min(history_ae_fd004.history["val_mae"]))
print("Best validation loss:",
      min(history_ae_fd004.history["val_loss"]))

Epoch 1/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 19s 43ms/step - loss: 3467.6345 - mae: 49.5847 - val_loss: 1735.1377 - val_mae: 36.7504
Epoch 2/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 14s 41ms/step - loss: 1748.2535 - mae: 36.8262 - val_loss: 1729.4330 - val_mae: 36.6833
Epoch 3/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 14s 41ms/step - loss: 1632.1046 - mae: 35.5667 - val_loss: 1212.8159 - val_mae: 30.3933
Epoch 4/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 15s 44ms/step - loss: 614.3642 - mae: 19.1949 - val_loss: 588.5265 - val_mae: 18.1942
Epoch 5/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 14s 42ms/step - loss: 482.8198 - mae: 16.5060 - val_loss: 678.8405 - val_mae: 20.4001
Epoch 6/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 14s 41ms/step - loss: 439.4523 - mae: 15.6317 - val_loss: 599.2021 - val_mae: 18.3707
Epoch 7/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 14s 41ms/step - loss: 414.2727 - mae: 15.1124 - val_loss: 569.0012 - val_mae: 17.4254
Epoch 8/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 14s 41ms/step - loss: 400.7331 - mae: 14.8272 - val_loss: 599.0925 - val

In [34]:
autoencoder_model.save("FD004_LSTM_Encoder_RUL125.keras")

import pandas as pd

history_ae_fd004_df = pd.DataFrame(
    history_ae_fd004.history
)

history_ae_fd004_df.to_csv(
    "FD004_LSTM_Encoder_Training_History.csv",
    index=False
)

print("FD004 Encoder model and history saved successfully")

FD004 Encoder model and history saved successfully


In [35]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# التنبؤ
y_pred_ae_fd004 = autoencoder_model.predict(
    X_test, verbose=0
).flatten()

y_true_ae_fd004 = np.asarray(y_test).flatten()

# MAE و RMSE
mae_ae_fd004 = mean_absolute_error(
    y_true_ae_fd004, y_pred_ae_fd004
)

rmse_ae_fd004 = np.sqrt(
    mean_squared_error(y_true_ae_fd004, y_pred_ae_fd004)
)

# NASA Score
errors = y_pred_ae_fd004 - y_true_ae_fd004

scores = np.where(
    errors < 0,
    np.exp(-errors / 13) - 1,
    np.exp(errors / 10) - 1
)

nasa_ae_fd004 = np.sum(scores)

print("FD004 LSTM Encoder Test Results")
print("--------------------------------")
print("MAE:", round(mae_ae_fd004, 4))
print("RMSE:", round(rmse_ae_fd004, 4))
print("NASA Score:", round(nasa_ae_fd004, 4))
print("Test Engines:", len(y_true_ae_fd004))

FD004 LSTM Encoder Test Results
--------------------------------
MAE: 24.3011
RMSE: 32.2536
NASA Score: 12951.2604
Test Engines: 248


In [36]:
import pandas as pd

fd004_encoder_results = pd.DataFrame([{
    "Dataset": "FD004",
    "Model": "LSTM_Encoder",
    "RUL_Cap": 125,
    "Seed": 42,
    "MAE": mae_ae_fd004,
    "RMSE": rmse_ae_fd004,
    "NASA_Score": nasa_ae_fd004,
    "Test_Engines": len(y_true_ae_fd004)
}])

fd004_encoder_results.to_csv(
    "FD004_Encoder_Test_Results.csv",
    index=False
)

print(fd004_encoder_results)
print("FD004 Encoder results saved successfully")

  Dataset         Model  RUL_Cap  Seed        MAE      RMSE    NASA_Score  \
0   FD004  LSTM_Encoder      125    42  24.301126  32.25362  12951.260399   

   Test_Engines  
0           248  
FD004 Encoder results saved successfully


In [37]:
set_seed(42)

encoder_fd001 = build_lstm_autoencoder()

early_stop_encoder_fd001 = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history_encoder_fd001 = encoder_fd001.fit(
    X_train_1,
    y_train_1,
    validation_data=(X_val_1, y_val_1),
    epochs=50,
    batch_size=128,
    callbacks=[early_stop_encoder_fd001],
    verbose=1
)

print("FD001 Encoder training completed")
print("Epochs:", len(history_encoder_fd001.history["loss"]))
print("Best validation MAE:",
      min(history_encoder_fd001.history["val_mae"]))
print("Best validation loss:",
      min(history_encoder_fd001.history["val_loss"]))

Epoch 1/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - loss: 4821.6099 - mae: 58.2816 - val_loss: 1469.7551 - val_mae: 31.0524
Epoch 2/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - loss: 715.6133 - mae: 21.7971 - val_loss: 275.6450 - val_mae: 13.6238
Epoch 3/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - loss: 341.0929 - mae: 14.6984 - val_loss: 204.5630 - val_mae: 11.0928
Epoch 4/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - loss: 262.4826 - mae: 12.4814 - val_loss: 166.3070 - val_mae: 9.7732
Epoch 5/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - loss: 244.4953 - mae: 11.9103 - val_loss: 148.4134 - val_mae: 8.8694
Epoch 6/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 6s 44ms/step - loss: 203.0442 - mae: 10.7916 - val_loss: 174.0466 - val_mae: 9.5386
Epoch 7/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - loss: 196.6973 - mae: 10.5929 - val_loss: 173.9292 - val_mae: 9.6840
Epoch 8/50
112/112 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - loss: 187.8976 - mae: 10.3017 - val_loss: 169.1857 - val_mae: 9.5764
Epo

In [38]:
encoder_fd001.save("FD001_LSTM_Encoder_RUL125.keras")

import pandas as pd

history_encoder_fd001_df = pd.DataFrame(
    history_encoder_fd001.history
)

history_encoder_fd001_df.to_csv(
    "FD001_LSTM_Encoder_Training_History.csv",
    index=False
)

print("FD001 Encoder model and history saved successfully")

FD001 Encoder model and history saved successfully


In [39]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# التنبؤ
y_pred_encoder_1 = encoder_fd001.predict(
    X_test_1, verbose=0
).flatten()

y_true_encoder_1 = np.asarray(y_test_1).flatten()

# حساب MAE و RMSE
mae_encoder_1 = mean_absolute_error(
    y_true_encoder_1, y_pred_encoder_1
)

rmse_encoder_1 = np.sqrt(
    mean_squared_error(y_true_encoder_1, y_pred_encoder_1)
)

# NASA Score
errors = y_pred_encoder_1 - y_true_encoder_1

scores = np.where(
    errors < 0,
    np.exp(-errors / 13) - 1,
    np.exp(errors / 10) - 1
)

nasa_encoder_1 = np.sum(scores)

print("FD001 LSTM Encoder Test Results")
print("--------------------------------")
print("MAE:", round(mae_encoder_1, 4))
print("RMSE:", round(rmse_encoder_1, 4))
print("NASA Score:", round(nasa_encoder_1, 4))
print("Test Engines:", len(y_true_encoder_1))

FD001 LSTM Encoder Test Results
--------------------------------
MAE: 10.4538
RMSE: 14.8817
NASA Score: 414.8726
Test Engines: 100


In [40]:
import pandas as pd

fd001_encoder_results = pd.DataFrame([{
    "Dataset": "FD001",
    "Model": "LSTM_Encoder",
    "RUL_Cap": 125,
    "Seed": 42,
    "MAE": mae_encoder_1,
    "RMSE": rmse_encoder_1,
    "NASA_Score": nasa_encoder_1,
    "Test_Engines": len(y_true_encoder_1)
}])

fd001_encoder_results.to_csv(
    "FD001_Encoder_Test_Results.csv",
    index=False
)

print(fd001_encoder_results)
print("FD001 Encoder results saved successfully")

  Dataset         Model  RUL_Cap  Seed        MAE       RMSE  NASA_Score  \
0   FD001  LSTM_Encoder      125    42  10.453788  14.881707  414.872634   

   Test_Engines  
0           100  
FD001 Encoder results saved successfully


In [9]:
def run_cap_experiment(cap):

    def make_windows_for_units(selected_units):

        X, y = [], []

        for unit in selected_units:

            engine = train_df[
                train_df["unit"] == unit
            ].sort_values("cycle")

            values = engine[FEATURE_COLS].to_numpy()
            labels = np.minimum(
                engine["RUL_linear"].to_numpy(),
                cap
            )

            for end in range(SEQ_LEN - 1, len(engine)):

                X.append(
                    values[end-SEQ_LEN+1:end+1]
                )
                y.append(labels[end])

        return (
            np.asarray(X, dtype=np.float32),
            np.asarray(y, dtype=np.float32)
        )

    Xtr, ytr = make_windows_for_units(train_units)
    Xva, yva = make_windows_for_units(val_units)

    print(
        f"Cap={cap} | Train={Xtr.shape}, "
        f"Validation={Xva.shape}"
    )

    set_seed(42)

    model = build_lstm()

    callback = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    history = model.fit(
        Xtr, ytr,
        validation_data=(Xva, yva),
        epochs=50,
        batch_size=128,
        callbacks=[callback],
        verbose=0
    )

    pred = np.maximum(
        model.predict(X_test, verbose=0).flatten(),
        0
    )

    error = pred - y_test

    return {
        "RUL cap": cap,
        "MAE": np.mean(np.abs(error)),
        "RMSE": np.sqrt(np.mean(error**2)),
        "NASA Score": nasa_score(y_test, pred),
        "Epochs": len(history.history["loss"])
    }

In [10]:
def run_cap_experiment(cap):

    def make_windows_for_units(selected_units):

        X, y = [], []

        for unit in selected_units:

            engine = train_df[
                train_df["unit"] == unit
            ].sort_values("cycle")

            values = engine[FEATURE_COLS].to_numpy()
            labels = np.minimum(
                engine["RUL_linear"].to_numpy(),
                cap
            )

            for end in range(SEQ_LEN - 1, len(engine)):

                X.append(
                    values[end-SEQ_LEN+1:end+1]
                )
                y.append(labels[end])

        return (
            np.asarray(X, dtype=np.float32),
            np.asarray(y, dtype=np.float32)
        )

    Xtr, ytr = make_windows_for_units(train_units)
    Xva, yva = make_windows_for_units(val_units)

    print(
        f"Cap={cap} | Train={Xtr.shape}, "
        f"Validation={Xva.shape}"
    )

    set_seed(42)

    model = build_lstm()

    callback = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    history = model.fit(
        Xtr, ytr,
        validation_data=(Xva, yva),
        epochs=50,
        batch_size=128,
        callbacks=[callback],
        verbose=0
    )

    pred = np.maximum(
        model.predict(X_test, verbose=0).flatten(),
        0
    )

    error = pred - y_test

    return {
        "RUL cap": cap,
        "MAE": np.mean(np.abs(error)),
        "RMSE": np.sqrt(np.mean(error**2)),
        "NASA Score": nasa_score(y_test, pred),
        "Epochs": len(history.history["loss"])
    }

In [11]:
# استخراج المتغيرات المطلوبة للمقارنة عالمياً وتفادي خطأ NameError
from sklearn.model_selection import train_test_split

# الحصول على المحركات وتقسيمها عالمياً
units = train_df["unit"].unique()
train_units, val_units = train_test_split(
    units, test_size=0.20, random_state=42
)

comparison = pd.DataFrame([
    run_cap_experiment(125),
    run_cap_experiment(150),
    run_cap_experiment(195)
])

print(comparison.round(4).to_string(index=False))

Cap=125 | Train=(43523, 30, 24), Validation=(10505, 30, 24)


NameError: name 'set_seed' is not defined

In [12]:
import os
import random
import numpy as np
import tensorflow as tf

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

print("set_seed function is ready")

set_seed function is ready


In [14]:
# إعادة تطبيق نفس التطبيع المستخدم في التجربة الأصلية
train_df_scaled = train_df.copy()

train_df_scaled[FEATURE_COLS] = scaler.transform(
    train_df_scaled[FEATURE_COLS]
)

print("Original feature means:")
print(train_df[FEATURE_COLS].mean().head())

print("\nScaled feature means:")
print(train_df_scaled[FEATURE_COLS].mean().head())

NameError: name 'scaler' is not defined

In [ ]:
comparison = pd.DataFrame([
    run_cap_experiment(125),
    run_cap_experiment(150),
    run_cap_experiment(195)
])

print(comparison.round(4).to_string(index=False))

In [ ]:
# مقارنة أول عينة من بيانات التدريب القديمة والجديدة

X_old = fd004_data[0]
y_old = fd004_data[1]

print("Old X shape:", X_old.shape)
print("New X shape:", Xtr.shape)

print("Old y first 10:", y_old[:10])
print("New y first 10:", ytr[:10])

print("Maximum feature difference:",
      np.max(np.abs(X_old - Xtr)))

print("Mean feature difference:",
      np.mean(np.abs(X_old - Xtr)))

print("First window difference:",
      np.max(np.abs(X_old[0] - Xtr[0])))

In [ ]:
# إعادة إنشاء بيانات FD004 بنفس المحركات والتطبيع

def rebuild_fd004_windows(selected_units):

    X, y = [], []

    for unit in selected_units:

        engine = train_df_scaled[
            train_df_scaled["unit"] == unit
        ].sort_values("cycle")

        values = engine[FEATURE_COLS].to_numpy()
        labels = engine["RUL"].to_numpy()

        for end in range(SEQ_LEN - 1, len(engine)):

            X.append(values[end-SEQ_LEN+1:end+1])
            y.append(labels[end])

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32)
    )


Xtr_check, ytr_check = rebuild_fd004_windows(train_units)
Xva_check, yva_check = rebuild_fd004_windows(val_units)

# المقارنة مع البيانات القديمة
X_old = fd004_data[0]
y_old = fd004_data[1]

print("===== SHAPES =====")
print("Old X:", X_old.shape)
print("New X:", Xtr_check.shape)
print("Old y:", y_old.shape)
print("New y:", ytr_check.shape)

print("\n===== LABELS =====")
print("Old y first 10:", y_old[:10])
print("New y first 10:", ytr_check[:10])

print("\n===== FEATURE DIFFERENCES =====")
print("Maximum difference:",
      np.max(np.abs(X_old - Xtr_check)))

print("Mean difference:",
      np.mean(np.abs(X_old - Xtr_check)))

print("First window difference:",
      np.max(np.abs(X_old[0] - Xtr_check[0])))

In [ ]:
# إعادة اختبار FD004 بسقف 125 مع تثبيت العشوائية

set_seed(42)

model_check = build_lstm()

early_stop_check = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history_check = model_check.fit(
    Xtr_check,
    ytr_check,
    validation_data=(Xva_check, yva_check),
    epochs=50,
    batch_size=128,
    callbacks=[early_stop_check],
    verbose=1
)

pred_check = np.maximum(
    model_check.predict(fd004_data[4]).flatten(),
    0
)

true_check = fd004_data[5]

error_check = pred_check - true_check

mae_check = np.mean(np.abs(error_check))
rmse_check = np.sqrt(np.mean(error_check**2))
score_check = nasa_score(true_check, pred_check)

print("\n===== REPRODUCIBILITY CHECK =====")
print("MAE:", mae_check)
print("RMSE:", rmse_check)
print("NASA Score:", score_check)
print("Epochs:", len(history_check.history["loss"]))

In [ ]:
import pandas as pd

results_so_far = pd.DataFrame([
    {
        "Dataset": "FD001",
        "MAE": 11.0069,
        "RMSE": 14.7038,
        "NASA_Score": 352.8176
    },
    {
        "Dataset": "FD004",
        "MAE": 24.2273,
        "RMSE": 31.5144,
        "NASA_Score": 9997.2473
    },
    {
        "Dataset": "FD004_Recheck",
        "MAE": 23.6127,
        "RMSE": 31.1272,
        "NASA_Score": 13500.1266
    }
])

results_so_far.to_csv(
    "/content/LSTM_initial_results.csv",
    index=False
)

print(results_so_far)
print("\nتم حفظ النتائج بنجاح.")

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import random

# تثبيت إعدادات التجربة
CAPS = [125, 150, 195]
SEEDS = [42, 123, 2026]

def make_windows_for_cap(selected_units, cap):

    X, y = [], []

    for unit in selected_units:

        engine = train_df_scaled[
            train_df_scaled["unit"] == unit
        ].sort_values("cycle")

        values = engine[FEATURE_COLS].to_numpy()

        labels = np.minimum(
            engine["RUL_linear"].to_numpy(),
            cap
        )

        for end in range(SEQ_LEN - 1, len(engine)):

            X.append(values[end-SEQ_LEN+1:end+1])
            y.append(labels[end])

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32)
    )


def run_one_experiment(cap, seed):

    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    Xtr, ytr = make_windows_for_cap(train_units, cap)
    Xva, yva = make_windows_for_cap(val_units, cap)

    model = build_lstm()

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    history = model.fit(
        Xtr, ytr,
        validation_data=(Xva, yva),
        epochs=50,
        batch_size=128,
        callbacks=[early_stop],
        verbose=0,
        shuffle=True
    )

    pred = np.maximum(
        model.predict(fd004_data[4], verbose=0).flatten(),
        0
    )

    actual = fd004_data[5]
    error = pred - actual

    return {
        "Cap": cap,
        "Seed": seed,
        "MAE": np.mean(np.abs(error)),
        "RMSE": np.sqrt(np.mean(error**2)),
        "NASA_Score": nasa_score(actual, pred),
        "Epochs": len(history.history["loss"])
    }


# تنفيذ التجارب التسع
all_results = []

for cap in CAPS:
    for seed in SEEDS:

        print(f"Running: RUL cap={cap}, seed={seed}")

        result = run_one_experiment(cap, seed)
        all_results.append(result)

        print(
            f"MAE={result['MAE']:.4f}, "
            f"RMSE={result['RMSE']:.4f}, "
            f"NASA={result['NASA_Score']:.2f}"
        )


# النتائج التفصيلية
details = pd.DataFrame(all_results)

# المتوسط والانحراف المعياري
summary = details.groupby("Cap").agg({
    "MAE": ["mean", "std"],
    "RMSE": ["mean", "std"],
    "NASA_Score": ["mean", "std"],
    "Epochs": ["mean"]
}).round(4)

print("\n===== DETAILED RESULTS =====")
print(details.to_string(index=False))

print("\n===== MEAN AND STANDARD DEVIATION =====")
print(summary)

# حفظ النتائج
details.to_csv("/content/FD004_RUL_Cap_Experiments.csv", index=False)
summary.to_csv("/content/FD004_RUL_Cap_Summary.csv")

print("\nتم حفظ نتائج التجارب التسع بنجاح.")

In [ ]:
from google.colab import files

files.download(
    "/content/FD004_RUL_Cap_Experiments.csv"
)

files.download(
    "/content/FD004_RUL_Cap_Summary.csv"
)

In [ ]:
from google.colab import files
files.download('/content/FD004_RUL_Cap_Summary.csv')


In [ ]:

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error

# إعدادات التجربة
SEQ_LEN = 30
CAPS = [125, 150, 195]
SEEDS = [42, 123, 2026]

# تحديد أسماء الأعمدة تلقائياً
unit_col = "unit" if "unit" in train_df_scaled.columns else "unit_nr"
cycle_col = "cycle" if "cycle" in train_df_scaled.columns else "time_cycles"

# تحديد أعمدة الخصائص
feature_cols = [
    c for c in train_df_scaled.columns
    if c.startswith("setting_") or c.startswith("sensor_")
]

if len(feature_cols) != 24:
    raise ValueError(
        f"Expected 24 features, found {len(feature_cols)}: {feature_cols}"
    )

def make_validation_sequences(df, selected_units, cap):
    X, y = [], []

    for uid in selected_units:
        g = df[df[unit_col] == uid].sort_values(cycle_col)
        features = g[feature_cols].to_numpy(dtype=np.float32)
        rul = np.minimum(
            g["RUL_linear"].to_numpy(dtype=np.float32), cap
        )

        for end in range(SEQ_LEN - 1, len(g)):
            X.append(features[end-SEQ_LEN+1:end+1])
            y.append(rul[end])

    return np.asarray(X), np.asarray(y)

def build_lstm():
    model = Sequential([
        Input(shape=(SEQ_LEN, 24)),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"]
    )
    return model

results = []

for cap in CAPS:
    Xtr, ytr = make_validation_sequences(
        train_df_scaled, train_units, cap
    )
    Xva, yva = make_validation_sequences(
        train_df_scaled, val_units, cap
    )

    print(f"\nRUL cap = {cap}")
    print("Train:", Xtr.shape, "Validation:", Xva.shape)

    for seed in SEEDS:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)

        model = build_lstm()

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True
        )

        history = model.fit(
            Xtr, ytr,
            validation_data=(Xva, yva),
            epochs=50,
            batch_size=128,
            callbacks=[early_stop],
            verbose=0
        )

        pred = model.predict(Xva, verbose=0).flatten()
        pred = np.maximum(pred, 0)

        mae = mean_absolute_error(yva, pred)
        rmse = np.sqrt(mean_squared_error(yva, pred))

        results.append({
            "RUL_Cap": cap,
            "Seed": seed,
            "Validation_MAE": mae,
            "Validation_RMSE": rmse,
            "Epochs": len(history.history["loss"])
        })

        print(
            f"Seed {seed}: "
            f"MAE={mae:.4f}, RMSE={rmse:.4f}"
        )

validation_results = pd.DataFrame(results)

validation_summary = (
    validation_results
    .groupby("RUL_Cap")
    .agg({
        "Validation_MAE": ["mean", "std"],
        "Validation_RMSE": ["mean", "std"],
        "Epochs": "mean"
    })
    .round(4)
)

display(validation_results)
display(validation_summary)

validation_results.to_csv(
    "/content/FD004_Validation_Cap_Experiments.csv",
    index=False
)

validation_summary.to_csv(
    "/content/FD004_Validation_Cap_Summary.csv"
)

print("تم حفظ ملفات نتائج التحقق بنجاح.")


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

def build_lstm():
    model = Sequential([
        tf.keras.Input(shape=(30, 24)),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"]
    )

    return model


def train_and_evaluate(dataset, subset):

    X_train, y_train, X_val, y_val, X_test, y_test, scaler = dataset

    model = build_lstm()

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=128,
        callbacks=[early_stop],
        verbose=1
    )

    predictions = model.predict(X_test).flatten()

    predictions = np.maximum(predictions, 0)

    errors = predictions - y_test

    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))

    # NASA asymmetric scoring function
    scores = np.where(
        errors < 0,
        np.exp(-errors / 13) - 1,
        np.exp(errors / 10) - 1
    )

    nasa_score = np.sum(scores)

    print(f"\\n===== {subset} RESULTS =====")
    print("MAE:", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("NASA Score:", round(nasa_score, 4))
    print("Epochs:", len(history.history["loss"]))

    return model, history, predictions, {
        "MAE": mae,
        "RMSE": rmse,
        "NASA_Score": nasa_score
    }


fd001_model, fd001_history, fd001_pred, fd001_metrics = train_and_evaluate(
    fd001_data, "FD001"
)

fd004_model, fd004_history, fd004_pred, fd004_metrics = train_and_evaluate(
    fd004_data, "FD004"
)

In [ ]:
def diagnose_predictions(y_true, y_pred, name):

    error = y_pred - y_true
    abs_error = np.abs(error)

    print("\n======", name, "======")
    print("Actual RUL min/max:",
          np.min(y_true), np.max(y_true))
    print("Predicted RUL min/max:",
          round(np.min(y_pred), 2),
          round(np.max(y_pred), 2))

    print("Actual RUL mean:",
          round(np.mean(y_true), 2))
    print("Predicted RUL mean:",
          round(np.mean(y_pred), 2))

    print("Mean prediction error:",
          round(np.mean(error), 2))

    print("Overestimations:",
          np.sum(error > 0))
    print("Underestimations:",
          np.sum(error < 0))

    print("Absolute errors > 30:",
          np.sum(abs_error > 30))

    print("Absolute error percentiles:",
          np.percentile(abs_error, [50, 75, 90, 95, 99]))


diagnose_predictions(
    fd001_data[5],
    fd001_pred,
    "FD001"
)

diagnose_predictions(
    fd004_data[5],
    fd004_pred,
    "FD004"
)

In [ ]:
# تشخيص أخطاء FD004 حسب قيمة RUL الحقيقية وطول سجل الاختبار

subset = "FD004"

test_df = data[(subset, "test")]
y_true = fd004_data[5]
y_pred = fd004_pred

errors = y_pred - y_true
abs_errors = np.abs(errors)

results = pd.DataFrame({
    "unit": sorted(test_df["unit"].unique()),
    "actual_RUL": y_true,
    "predicted_RUL": y_pred,
    "absolute_error": abs_errors
})

results["RUL_group"] = np.where(
    results["actual_RUL"] > 125,
    "RUL > 125",
    "RUL <= 125"
)

print("===== Errors by actual RUL =====")
print(results.groupby("RUL_group").agg(
    engines=("unit", "count"),
    MAE=("absolute_error", "mean"),
    RMSE=("absolute_error",
          lambda x: np.sqrt(np.mean(x**2))),
    max_error=("absolute_error", "max")
).round(3))

lengths = test_df.groupby("unit")["cycle"].count()

print("\n===== Test sequence lengths =====")
print("Engines with fewer than 30 cycles:", (lengths < 30).sum())
print("Shortest sequence:", lengths.min())
print("Longest sequence:", lengths.max())

print("\n===== Largest 15 prediction errors =====")
print(
    results.sort_values("absolute_error", ascending=False)
    .head(15).round(2).to_string(index=False)
)

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import random

# تثبيت العشوائية لتسهيل المقارنة
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

# بيانات FD004 المجهزة سابقًا
X_train, y_train_125, X_val, y_val_125, X_test, y_test, scaler = fd004_data

# استعادة RUL الخطي قبل القص
train_df = data[("FD004", "train")].copy()

# التأكد من وجود العمود
assert "RUL_linear" in train_df.columns

def nasa_score(y_true, y_pred):
    error = y_pred - y_true
    return np.sum(np.where(
        error < 0,
        np.exp(-error / 13) - 1,
        np.exp(error / 10) - 1
    ))

def run_cap_experiment(cap):

    # إعادة بناء تسميات التدريب بالسقف المحدد
    y_new = []

    for unit in sorted(train_df["unit"].unique()):
        engine = train_df[train_df["unit"] == unit].sort_values("cycle")
        labels = np.minimum(engine["RUL_linear"].to_numpy(), cap)

        y_new.extend(labels[29:])

    y_new = np.asarray(y_new, dtype=np.float32)

    assert len(y_new) == len(X_train)

    set_seed(42)

    model = build_lstm()

    callback = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, y_new,
        validation_data=(X_val, y_val_125),
        epochs=50,
        batch_size=128,
        callbacks=[callback],
        verbose=0
    )

    pred = np.maximum(model.predict(X_test, verbose=0).flatten(), 0)

    mae = np.mean(np.abs(pred - y_test))
    rmse = np.sqrt(np.mean((pred - y_test)**2))
    score = nasa_score(y_test, pred)

    return {
        "RUL cap": cap,
        "MAE": mae,
        "RMSE": rmse,
        "NASA Score": score,
        "Epochs": len(history.history["loss"])
    }

comparison = pd.DataFrame([
    run_cap_experiment(125),
    run_cap_experiment(150),
    run_cap_experiment(195)
])

print(comparison.round(4).to_string(index=False))

In [ ]:
print("X_train:", X_train.shape)
print("y_train original:", y_train_125.shape)
print("X_val:", X_val.shape)
print("y_val original:", y_val_125.shape)

print("Total training engines:", len(train_units))
print("Training engines:", len(train_units))
print("Validation engines:", len(val_units))

print("Total FD004 engines:", train_df["unit"].nunique())

## ما الذي نحتاجه منك بعد التشغيل؟

إذا انتهت الخلايا دون خطأ، أرسل لي:
1. صورة أو نسخًا من مخرجات الخلية 3 والخلية 4.
2. إذا ظهر خطأ، أرسل نص الخطأ كاملًا.

**مهم:** لا نبدأ تدريب LSTM قبل التأكد من تطابق الملفات وأعداد محركات الاختبار مع ملفات RUL. وبعد نجاح الفحص، ننتقل إلى تجهيز بيانات RUL ثم تدريب النموذج، وبعدها LSTM Autoencoder وقياس FP32 مقابل INT8.
